# Logistic Regression con validacion LOSO

Este notebook implementa el **Experimento A**: features derivadas de HR/R-R y ECG.

El objetivo es evaluar la generalizacion a un trabajador no visto durante el entrenamiento mediante Leave-One-Subject-Out Cross-Validation (LOSO). El target se crea dentro de cada fold usando unicamente `FatigueIndex` del conjunto TRAIN.

## Regla de no leakage

En cada fold se siguen estos pasos:

1. Un trabajador queda como TEST y los otros cuatro forman TRAIN.
2. Se calculan P33 y P66 usando solo `FatigueIndex` de TRAIN.
3. Se crean las etiquetas `Low`, `Medium` y `High` con esos percentiles.
4. Se aplica exactamente la misma categorizacion al TEST.
5. Cualquier imputador o scaler se ajusta solo con TRAIN.

Las entradas son las columnas `_z`, ya normalizadas respecto a la baseline individual de cada trabajador. No se usa `FatigueIndex` como feature.

In [1]:
from pathlib import Path

import json
import joblib
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'nootebooks' else Path.cwd()
DATA_DIR = ROOT_DIR
BASELINE_WINDOW_COUNT = 15
TARGET_COLUMN_CANDIDATES = ['FatigueIndex', 'fatigue_index']
LABELS = ['Low', 'Medium', 'High']
C_GRID = [100.0, 300.0, 1000.0, 3000.0, 10000.0]

print(f'Directorio de datos: {DATA_DIR}')

Directorio de datos: c:\Users\Carlo\Desktop\owncloud 2025-08-11 ECG\new


## Carga de datos

Cada CSV corresponde a un trabajador. El nombre de la carpeta (`W00`, ..., `W08`) se conserva como identificador para construir los folds, pero nunca entra como feature del modelo.

In [2]:
worker_files = {
    worker_dir.name: worker_dir / 'PROCESSED' / 'combined_features_1min.csv'
    for worker_dir in sorted(DATA_DIR.glob('W[0-9][0-9]'))
    if (worker_dir / 'PROCESSED' / 'combined_features_1min.csv').exists()
}
if len(worker_files) < 2:
    raise ValueError(f'Se necesitan al menos 2 trabajadores procesados y se encontraron {len(worker_files)}: {list(worker_files)}')

data = {}
for worker, path in worker_files.items():
    frame = pd.read_csv(path)
    target_matches = [column for column in TARGET_COLUMN_CANDIDATES if column in frame.columns]
    if len(target_matches) != 1:
        raise ValueError(f'{worker}: no se encontro exactamente una columna FatigueIndex/fatigue_index')
    frame = frame.rename(columns={target_matches[0]: 'FatigueIndex'})
    frame['Trabajador'] = worker
    data[worker] = frame

feature_columns = [
    'ECG_energy_z',
    'ECG_mean_z',
    'ECG_missing_peaks_z',
    'ECG_range_z',
    'ECG_samp_ent_z',
    'ECG_std_z',
    'HR_max_z',
    'HR_mean_z',
    'HR_min_z',
    'RMSSD_z',
    'SDNN_z',
    'pNN50_z',
    'HR_mean_ultimos_5min_z',
    'cambio_HR_vs_baseline_z',
    'tendencia_HR_z',
    'cambio_RMSSD_z',
]
missing_features = [
    column for column in feature_columns
    if column not in data[next(iter(data))].columns
]
if missing_features:
    raise ValueError(f'Faltan features requeridas: {missing_features}')
if not feature_columns:
    raise ValueError('No se encontraron columnas de entrada terminadas en _z')

print('Trabajadores:', list(data))
print('Features del Experimento A:', feature_columns)
print('Ventanas:', {worker: len(frame) for worker, frame in data.items()})

Trabajadores: ['W00', 'W01', 'W02', 'W03', 'W04', 'W05', 'W06', 'W07', 'W08']
Features del Experimento A: ['ECG_energy_z', 'ECG_mean_z', 'ECG_missing_peaks_z', 'ECG_range_z', 'ECG_samp_ent_z', 'ECG_std_z', 'HR_max_z', 'HR_mean_z', 'HR_min_z', 'RMSSD_z', 'SDNN_z', 'pNN50_z', 'HR_mean_ultimos_5min_z', 'cambio_HR_vs_baseline_z', 'tendencia_HR_z', 'cambio_RMSSD_z']
Ventanas: {'W00': 104, 'W01': 637, 'W02': 603, 'W03': 407, 'W04': 562, 'W05': 74, 'W06': 35, 'W07': 67, 'W08': 16}


## Funciones de target y evaluacion

Los percentiles se calculan en cada fold y solo con el vector de `FatigueIndex` de TRAIN. Los valores faltantes del target no se usan para calcular percentiles ni para entrenar/evaluar ese fold.

In [3]:
def make_labels(fatigue_values, p33, p66):
    return pd.cut(
        fatigue_values,
        bins=[-np.inf, p33, p66, np.inf],
        labels=LABELS,
        right=False,
    ).astype(object)


def get_train_percentiles(train_frame):
    train_fatigue = pd.to_numeric(train_frame['FatigueIndex'], errors='coerce').dropna()
    if train_fatigue.empty:
        raise ValueError('TRAIN no contiene valores validos de FatigueIndex')
    return np.percentile(train_fatigue, [33, 66])


def build_model(C=1.0):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            C=C,
            class_weight='balanced',
            max_iter=2000,
            random_state=42,
        )),
    ])


def optimize_model(X_train, y_train):
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    search = GridSearchCV(
        estimator=build_model(),
        param_grid={'classifier__C': C_GRID},
        scoring='balanced_accuracy',
        cv=inner_cv,
        refit=True,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
    return search

## Experimento A

Features utilizadas: todas las columnas `_z` disponibles, que corresponden a features derivadas de HR/R-R y ECG.

La matriz de confusion utiliza siempre el orden `Low`, `Medium`, `High`.

In [4]:
fold_results = []
confusion_matrices = {}

for test_worker in sorted(data):
    train_workers = [worker for worker in sorted(data) if worker != test_worker]
    train_frame = pd.concat([data[worker] for worker in train_workers], ignore_index=True)
    test_frame = data[test_worker].copy()

    p33, p66 = get_train_percentiles(train_frame)
    train_target = make_labels(
        pd.to_numeric(train_frame['FatigueIndex'], errors='coerce'), p33, p66
    )
    test_target = make_labels(
        pd.to_numeric(test_frame['FatigueIndex'], errors='coerce'), p33, p66
    )

    train_mask = train_target.notna()
    test_mask = test_target.notna()
    X_train = train_frame.loc[train_mask, feature_columns]
    y_train = train_target.loc[train_mask]
    X_test = test_frame.loc[test_mask, feature_columns]
    y_test = test_target.loc[test_mask]

    if y_train.nunique() < 2:
        raise ValueError(f'{test_worker}: TRAIN tiene menos de dos clases')
    if y_test.empty:
        raise ValueError(f'{test_worker}: TEST no contiene FatigueIndex valido')

    search = optimize_model(X_train, y_train)
    y_pred = search.predict(X_test)

    confusion_matrices[test_worker] = confusion_matrix(
        y_test, y_pred, labels=LABELS
    )
    matrix = confusion_matrices[test_worker]
    class_recall = np.diag(matrix) / matrix.sum(axis=1)
    fold_results.append({
        'Test_worker': test_worker,
        'Train_workers': ', '.join(train_workers),
        'P33_train': p33,
        'P66_train': p66,
        'N_train': len(y_train),
        'N_test': len(y_test),
        'Best_C': search.best_params_['classifier__C'],
        'Recall_Low': class_recall[0],
        'Recall_Medium': class_recall[1],
        'Recall_High': class_recall[2],
        'Accuracy': accuracy_score(y_test, y_pred),
        'Balanced_Accuracy': balanced_accuracy_score(y_test, y_pred),
        'Macro_F1': f1_score(y_test, y_pred, labels=LABELS, average='macro', zero_division=0),
    })

fold_results_df = pd.DataFrame(fold_results)
mean_row = {column: np.nan for column in fold_results_df.columns}
mean_row['Test_worker'] = 'Media'
mean_row['Train_workers'] = 'Promedio de los folds'
for metric in ['P33_train', 'P66_train', 'N_train', 'N_test', 'Best_C', 'Recall_Low', 'Recall_Medium', 'Recall_High', 'Accuracy', 'Balanced_Accuracy', 'Macro_F1']:
    mean_row[metric] = fold_results_df[metric].mean()
fold_results_with_mean = pd.concat(
    [fold_results_df, pd.DataFrame([mean_row])], ignore_index=True
)
fold_results_with_mean

,Test_worker,Train_workers,P33_train,P66_train,N_train,N_test,Best_C,Recall_Low,Recall_Medium,Recall_High,Accuracy,Balanced_Accuracy,Macro_F1
0,W00,"W01, W02, W03, W04, W05, W06, W07, W08",-0.191350,1.838830,2401.000000,104.000000,1000.000000,0.964286,0.750000,0.500000,0.894231,0.738095,0.719489
1,W01,"W00, W02, W03, W04, W05, W06, W07, W08",-0.344569,1.613752,1868.000000,637.000000,10000.000000,0.825688,0.982143,0.588710,0.802198,0.798847,0.812360
2,W02,"W00, W01, W03, W04, W05, W06, W07, W08",-0.350388,1.680866,1902.000000,603.000000,3000.000000,0.827957,0.799308,0.990950,0.873964,0.872738,0.872520
3,W03,"W00, W01, W02, W04, W05, W06, W07, W08",-0.314058,1.003435,2098.000000,407.000000,3000.000000,0.984615,0.571429,0.945860,0.926290,0.833968,0.800205
4,W04,"W00, W01, W02, W03, W05, W06, W07, W08",0.163252,2.827953,1943.000000,562.000000,10000.000000,0.794411,0.574074,0.714286,0.772242,0.694257,0.526688
5,W05,"W00, W01, W02, W03, W04, W06, W07, W08",-0.233753,1.773938,2431.000000,74.000000,10000.000000,0.863636,0.848485,0.842105,0.851351,0.851409,0.856385
6,W06,"W00, W01, W02, W03, W04, W05, W07, W08",-0.227075,1.754840,2470.000000,35.000000,3000.000000,0.800000,0.500000,0.666667,0.714286,0.655556,0.659524
7,W07,"W00, W01, W02, W03, W04, W05, W06, W08",-0.246540,1.674883,2438.000000,67.000000,10000.000000,1.000000,0.111111,0.341463,0.358209,0.484192,0.325589
8,W08,"W00, W01, W02, W03, W04, W05, W06, W07",-0.230620,1.740030,2489.000000,16.000000,3000.000000,1.000000,0.600000,0.833333,0.812500,0.811111,0.811966
9,Media,Promedio de los folds,-0.219456,1.767614,2226.666667,278.333333,5888.888889,0.895622,0.637394,0.713708,0.778363,0.748908,0.709414


In [5]:
print('Matrices de confusion por trabajador TEST:')
for worker, matrix in confusion_matrices.items():
    print(f'\nTEST = {worker}')
    print(pd.DataFrame(matrix, index=LABELS, columns=LABELS))

metrics = ['Accuracy', 'Balanced_Accuracy', 'Macro_F1']
summary_df = pd.DataFrame({
    'Metric': metrics,
    'Mean': [fold_results_df[metric].mean() for metric in metrics],
    'Std': [fold_results_df[metric].std(ddof=1) for metric in metrics],
})
print('Media y desviacion estandar de las metricas:')
display(summary_df)

aggregate_confusion = np.sum(
    np.stack([confusion_matrices[worker] for worker in sorted(confusion_matrices)]),
    axis=0,
)
print('Matriz de confusion agregada de los cinco folds:')
display(pd.DataFrame(aggregate_confusion, index=LABELS, columns=LABELS))

class_support = aggregate_confusion.sum(axis=1)
predicted_support = aggregate_confusion.sum(axis=0)
class_metrics_df = pd.DataFrame({
    'Support': class_support,
    'Recall': np.diag(aggregate_confusion) / class_support,
    'Precision': np.diag(aggregate_confusion) / predicted_support,
}, index=LABELS)
print('Metricas agregadas por clase:')
display(class_metrics_df.round(3))

distribution_rows = []
for result in fold_results:
    worker = result['Test_worker']
    fatigue_values = pd.to_numeric(data[worker]['FatigueIndex'], errors='coerce')
    labels = make_labels(fatigue_values, result['P33_train'], result['P66_train'])
    counts = labels.value_counts().reindex(LABELS, fill_value=0)
    distribution_rows.append({
        'Test_worker': worker,
        'Low': int(counts['Low']),
        'Medium': int(counts['Medium']),
        'High': int(counts['High']),
        'Total_valid': int(counts.sum()),
    })
print('Distribucion de clases en cada trabajador TEST:')
display(pd.DataFrame(distribution_rows))

Matrices de confusion por trabajador TEST:

TEST = W00
        Low  Medium  High
Low      81       3     0
Medium    2       6     0
High      0       6     6

TEST = W01
        Low  Medium  High
Low      90      19     0
Medium    5     275     0
High      0     102   146

TEST = W02
        Low  Medium  High
Low      77      15     1
Medium    8     231    50
High      0       2   219

TEST = W03
        Low  Medium  High
Low      64       1     0
Medium   12      16     0
High      2      15   297

TEST = W04
        Low  Medium  High
Low     398      97     6
Medium   14      31     9
High      1       1     5

TEST = W05
        Low  Medium  High
Low      19       3     0
Medium    4      28     1
High      0       3    16

TEST = W06
        Low  Medium  High
Low      16       4     0
Medium    2       3     1
High      2       1     6

TEST = W07
        Low  Medium  High
Low       8       0     0
Medium   15       2     1
High     14      13    14

TEST = W08
        Low  Medi

,Metric,Mean,Std
0,Accuracy,0.778363,0.170360
1,Balanced_Accuracy,0.748908,0.123152
2,Macro_F1,0.709414,0.180576


Matriz de confusion agregada de los cinco folds:


,Low,Medium,High
Low,758,142,7
Medium,62,595,64
High,19,144,714


Metricas agregadas por clase:


,Support,Recall,Precision
Low,907,0.836,0.903
Medium,721,0.825,0.675
High,877,0.814,0.910


Distribucion de clases en cada trabajador TEST:


,Test_worker,Low,Medium,High,Total_valid
0,W00,84,8,12,104
1,W01,109,280,248,637
2,W02,93,289,221,603
3,W03,65,28,314,407
4,W04,501,54,7,562
5,W05,22,33,19,74
6,W06,20,6,9,35
7,W07,8,18,41,67
8,W08,5,5,6,16


In [ ]:
c_by_fold = '; '.join(
    f'{row.Test_worker}: {row.Best_C:g}'
    for row in fold_results_df.itertuples(index=False)
)
final_c = (
    str(final_search.best_params_['classifier__C'])
    if 'final_search' in globals()
    else 'Pendiente de entrenar'
)

comparison_df = pd.DataFrame({
    'Modelo': [
        'Original sin variables temporales',
        'Temporal original',
        'Optimizado anterior',
        'Optimizado con cuadrícula ampliada',
    ],
    'class_weight': ['None', 'None', 'balanced', 'balanced'],
    'C por fold': [
        '1.0',
        '1.0',
        '100 en los 5 folds',
        c_by_fold,
    ],
    'C final': [
        '1.0',
        '1.0',
        '100',
        final_c,
    ],
    'Accuracy': [
        0.781303,
        0.783572,
        0.844313,
        fold_results_df['Accuracy'].mean(),
    ],
    'Balanced Accuracy': [
        0.679752,
        0.692432,
        0.702434,
        fold_results_df['Balanced_Accuracy'].mean(),
    ],
    'Macro-F1': [
        0.652589,
        0.667675,
        0.681970,
        fold_results_df['Macro_F1'].mean(),
    ],
})

for column in ['Accuracy', 'Balanced Accuracy', 'Macro-F1']:
    comparison_df[column] = comparison_df[column].map(
        lambda value: f'{value:.2%}'
    )

display(comparison_df)

NameError: name 'final_search' is not defined

## Modelo final y exportacion

La evaluacion LOSO anterior mide la generalizacion dejando un trabajador fuera en cada fold. Despues, este bloque reentrena un modelo final con todos los trabajadores y todas las ventanas validas. Este modelo final es el que se utilizara para la aplicacion.

El modelo y el preprocesamiento completo se guardan en formato `joblib`, el modelo para Expo/React Native en formato `ONNX`, y la configuracion en un archivo `JSON`. En este notebook se guardan en `new/models/tempvar_opt/`.

In [ ]:
# Reentrenamiento final con todos los trabajadores y optimizacion interna de C
all_frame = pd.concat([data[worker] for worker in sorted(data)], ignore_index=True)
all_fatigue = pd.to_numeric(all_frame['FatigueIndex'], errors='coerce')
final_p33, final_p66 = np.percentile(all_fatigue.dropna(), [33, 66])
all_target = make_labels(all_fatigue, final_p33, final_p66)
all_mask = all_target.notna()
X_all = all_frame.loc[all_mask, feature_columns]
y_all = all_target.loc[all_mask]

final_search = optimize_model(X_all, y_all)
final_model = final_search.best_estimator_

models_dir = ROOT_DIR / 'models' / 'tempvar_opt'
models_dir.mkdir(parents=True, exist_ok=True)
joblib_path = models_dir / 'final_logistic_regression.joblib'
onnx_path = models_dir / 'final_logistic_regression.onnx'
metadata_path = models_dir / 'final_logistic_regression_metadata.json'

joblib.dump(final_model, joblib_path)
onnx_model = convert_sklearn(
    final_model,
    initial_types=[('features', FloatTensorType([None, len(feature_columns)]))],
)
onnx_path.write_bytes(onnx_model.SerializeToString())

metadata = {
    'model_type': 'LogisticRegression',
    'validation': 'LOSO_with_inner_C_search',
    'training_workers': sorted(data),
    'feature_columns': feature_columns,
    'feature_count': len(feature_columns),
    'labels': LABELS,
    'class_weight': 'balanced',
    'C_grid': C_GRID,
    'selected_C_final': float(final_search.best_params_['classifier__C']),
    'p33_train_all_workers': float(final_p33),
    'p66_train_all_workers': float(final_p66),
    'target_column': 'FatigueIndex',
    'temporal_features': [
        'HR_mean_ultimos_5min_z',
        'cambio_HR_vs_baseline_z',
        'tendencia_HR_z',
        'cambio_RMSSD_z',
    ],
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print(f'Modelo final entrenado con {len(y_all)} ventanas y {len(feature_columns)} features')
print(f'C final seleccionado: {final_search.best_params_["classifier__C"]}')
print(f'Guardado en: {models_dir}')

Modelo final entrenado con 2313 ventanas y 16 features
C final seleccionado: 10000.0
Guardado en: c:\Users\Carlo\Desktop\owncloud 2025-08-11 ECG\new\models\tempvar_opt
